# PaisaPlan: LoRA fine-tuning of Llama 3.2 3B Instruct

Run the cells **top to bottom**. Runtime > Change runtime type > **T4 GPU** first.
Upload `train.jsonl` and `val.jsonl` in step 2.

## 1. Install

In [ ]:
%%capture
!pip install unsloth

## 2. Upload the data (choose train.jsonl AND val.jsonl)

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

## 3. Load the pre-trained model in 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

## 4. Attach LoRA adapters (only these small weights get trained)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 5. Load and format the dataset

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template = "llama-3.1")
ds = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})

def fmt(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]}

ds = ds.map(fmt, batched=True)
print(ds)
print(ds["train"][0]["text"])

## 6. Train (about 10-20 min on a T4). Watch that validation loss goes down.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds["train"],
    eval_dataset = ds["validation"],
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        warmup_steps = 10,
        logging_steps = 5,
        eval_strategy = "steps",
        eval_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
# train only on the assistant's answer, not on the prompt
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)
trainer.train()

## 7. Loss curve (save this for the report)

In [ ]:
import matplotlib.pyplot as plt
logs = trainer.state.log_history
tr = [(l["step"], l["loss"]) for l in logs if "loss" in l]
ev = [(l["step"], l["eval_loss"]) for l in logs if "eval_loss" in l]
plt.plot(*zip(*tr), label="train")
plt.plot(*zip(*ev), label="validation", marker="o")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("PaisaPlan LoRA fine-tuning")
plt.savefig("loss_curve.png", dpi=150)
plt.show()

## 8. Quick test on new messages

In [ ]:
EXTRACT_SYS = "You are the input parser for PaisaPlan, a budget planner. Convert the user's message into JSON with exactly these keys: income, fixed_costs, priorities, savings_goal, change_request. Output only the JSON. Use null (or an empty list) when something is not mentioned. Never invent numbers."
EXPLAIN_SYS = "You are the explainer for PaisaPlan, a budget planner. You receive solver results as JSON. Explain them in plain, friendly language. Use only numbers that appear in the JSON and never calculate new ones. Mark facts as [CERTAIN] and projections as [ESTIMATE: reason]."

FastLanguageModel.for_inference(model)

def ask(system, user):
    msgs = [{"role":"system","content":system},{"role":"user","content":user}]
    ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=300, do_sample=False)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

for m in ["my stipend is 22k, hostel 7000, wanna save 6000 in 3 months",
          "what if I spend 1500 less on travel",
          "salary thirty five thousand, fees 4k, food first then savings"]:
    print(m, "->", ask(EXTRACT_SYS, m), "\n")

## 9. Save adapter and export GGUF for Ollama

In [ ]:
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")
model.save_pretrained_gguf("paisaplan_gguf", tokenizer, quantization_method = "q4_k_m")

## 10. Copy everything to Google Drive (the GGUF is ~2 GB, so Drive is easier than a browser download)

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/paisaplan
!cp -r paisaplan_gguf lora_adapter loss_curve.png /content/drive/MyDrive/paisaplan/
files.download('loss_curve.png')